# Laya-1.2 — Fast Tiled Ternary V2 (FP16 Tensor Cores + Activation-Aware G128)

**Source:** [vtava/Laya-1.2-speedup-standalone-fp16](https://huggingface.co/vtava/Laya-1.2-speedup-standalone-fp16)  
**Experiment repository:** [TinyCeNN-LM](https://github.com/vtavakkoli/TinyCeNN-LM)

### Why this V2 experiment?

The first Laya cascade run on **Tesla T4** converted four projections, reduced serialized FP16 model state from **803.05 to 786.27 MiB** (2.1%), and reached **96.875% independent-test agreement (124/128)**, just below its 97% quality gate. The old Triton backend was **2.31× slower** than FP16 at B4 × T512, while cached FP16 reconstruction was almost as fast as the original.

V2 targets the *cause* of the slowdown: the earlier kernel repeatedly performed 128-wide dots and FP32 partial group rescaling. New **tiled packed-GEMM** loads 64/128/256/512-wide chunks, unpacks 2-bit signed ternary codes in the kernel, applies FP16 G128 group scales *before* FP16 tensor-core dot operations, and accumulates in FP32. Whether it is faster is **measured**, not assumed.

It also tests activation-aware **closed-form ridge fitting of group scales** while keeping ternary codes fixed, using calibration-only activations, and a larger disjoint train-split development/audit partition. The actual test split is untouched until the model selection finishes.

**Interpretation:** G128 stores two-bit codes (-1,0,+1) plus 16-bit scale / 128 weights (approximately 2.125 bits per converted weight). This is **not** physical 1.58-bit storage, trained BitNet or original proprietary Bonsai.

**Suggested initial run:** L4 if available, T4 also supported; use six layers first. Keep **PUBLISH disabled** — this notebook does not upload or overwrite model weights.


In [ ]:
#@title 1. Install dependencies
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" "triton>=3.0,<4" pandas matplotlib


In [ ]:
#@title 2. Experiment configuration + Hugging Face checkpoint download
import os, sys, gc, json, math, time, random, importlib.util, inspect, shutil, zipfile
from pathlib import Path
import torch, numpy as np, pandas as pd
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download
from IPython.display import display

SOURCE_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
FIRST_LAYER=0 #@param {type:"integer"}
MAX_LAYERS=6 #@param {type:"integer"}
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=900 #@param {type:"integer"}
CAL_DECISIONS=48 #@param {type:"integer"}
DEV_DECISIONS=192 #@param {type:"integer"}
AUDIT_DECISIONS=192 #@param {type:"integer"}
OFFICIAL_TEST_DECISIONS=200 #@param {type:"integer"}
MIN_DEV_AGREEMENT=0.97 #@param {type:"number"}
MIN_AUDIT_AGREEMENT=0.97 #@param {type:"number"}
MAX_MEAN_JS=0.015 #@param {type:"number"}
CALIBRATION_TOKENS_PER_PROJECTION=384 #@param {type:"integer"}
RIDGE_RELATIVE=0.03 #@param {type:"number"}
ENABLE_SCALE_REFINEMENT=True #@param {type:"boolean"}
MIN_LINEAR_ELEMENTS=4096 #@param {type:"integer"}
BENCH_REPEATS=8 #@param {type:"integer"}
SEED=2031
OUT=Path("/content/Laya_Tiled_Ternary_V2")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
assert torch.cuda.get_device_capability()[0]>=7, "Requires a tensor-core GPU, e.g. T4/L4"
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN=os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("Trying public Hugging Face access; for private models add Colab Secret HF_TOKEN.")
ROOT=Path(snapshot_download(repo_id=SOURCE_REPO,token=HF_TOKEN))
assert (ROOT/"model.safetensors").is_file() and (ROOT/"config.json").is_file()
META=json.loads((ROOT/"config.json").read_text())
assert META.get("format")=="laya-1p2-full-standalone-v1", META
assert META.get("dtype_on_disk")=="float16", "This V2 notebook requires true FP16 checkpoint."
print("GPU:",torch.cuda.get_device_name(0))
print("Standalone:",META.get("architecture"), "| weights FP16")


In [ ]:
#@title 3. Load the exact LayaStandalone checkpoint
def import_py(path,name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    if spec is None or spec.loader is None:raise ImportError(path)
    m=importlib.util.module_from_spec(spec)
    sys.modules[name]=m;spec.loader.exec_module(m)
    return m

P=ROOT/"modeling_laya_standalone.py"
if not P.is_file():
    raise FileNotFoundError("Expected modeling_laya_standalone.py alongside this Laya FP16 checkpoint.")
source=import_py(P,"_laya_fp16_source_v2")
assert callable(getattr(source,"load_model",None))
reference,tokenizer=source.load_model(ROOT,device="cpu",local_files_only=True,compile_head=False)
reference=reference.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
reference.encoder.config.reference_compile=False
assert hasattr(reference.encoder,"layers")
LAYER_COUNT=len(reference.encoder.layers)
assert 0<=FIRST_LAYER<LAYER_COUNT and MAX_LAYERS>=1
print("Original encoder layers:",LAYER_COUNT)
print("Parameters:",f"{sum(p.numel() for p in reference.parameters()):,}")
print("Scope: typed decision option scores; no action head.")


In [ ]:
#@title 4. Load pinned V1/V2 kernels; test ternary reconstruction and tiled CUDA correctness
from urllib.request import urlopen
PIN_V1="ed1ede5f4fc6430b78fab6b66e87d4f4053f2469"
PIN_V2="d1eddf3ec8b5c91fb89dbc0141f585b10b30600d"
RAW="https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/"
def source_file(commit,name):
    local=Path("/content")/name
    with urlopen(f"{RAW}{commit}/src/tinycenn_lm/{name}",timeout=90) as resp:
        local.write_bytes(resp.read())
    assert local.stat().st_size>4000
    return local
old=import_py(source_file(PIN_V1,"laya_fp16_ternary_cascade.py"),
              "laya_fp16_ternary_cascade")
fast=import_py(source_file(PIN_V2,"laya_ternary_tiled_v2.py"),
               "laya_ternary_tiled_v2")
assert fast.triton is not None

# Numerical correctness against identical weights dequantized to dense FP16.
probe=nn.Linear(768,768,bias=True,device="cuda",dtype=torch.float16)
quant=fast.TiledTernaryLinear.from_linear(probe,tile_k=128)
inp=torch.randn(128,768,device="cuda",dtype=torch.float16)
with torch.inference_mode():
    expected=F.linear(inp,quant.dense_weight(),quant.bias).float()
GOOD_BK=[];KERNEL_ERRORS={}
for bk in (64,128,256,512):
    quant.tile_k=bk
    try:
        with torch.inference_mode():
            pred=quant(inp).float()
        error=(pred-expected).abs()
        torch.testing.assert_close(pred,expected,atol=0.07,rtol=0.035)
        GOOD_BK.append(bk)
        print(f"BK={bk}: correct | max error {float(error.max()):.5f}, mean error {float(error.mean()):.6f}")
    except Exception as exc:
        KERNEL_ERRORS[str(bk)]=repr(exc)
        print(f"BK={bk}: FAILED:",str(exc)[:320])
assert GOOD_BK, "No fused-tile kernel passed the CUDA numerical preflight"
del probe,quant,inp,expected
gc.collect();torch.cuda.empty_cache()
print("Correct tiled kernel candidates:",GOOD_BK)


In [ ]:
#@title 5. Fair isolated FP16 / legacy packed / tiled / cached FP16 GPU microbenchmarks
@torch.inference_mode()
def timer(fn,repeats=BENCH_REPEATS,warmup=4):
    for _ in range(warmup):fn()
    torch.cuda.synchronize()
    timings=[]
    for _ in range(repeats):
        start=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        start.record();fn();end.record()
        torch.cuda.synchronize()
        timings.append(start.elapsed_time(end))
    return float(np.median(timings))

shapes=[(768,768,128),(768,3072,512),(3072,768,512)]
micro=[]
for indim,outdim,m in shapes:
    base=nn.Linear(indim,outdim,device="cuda",dtype=torch.float16)
    x=torch.randn(m,indim,device="cuda",dtype=torch.float16)
    module=fast.TiledTernaryLinear.from_linear(base,backend="triton_tiled")
    fp16ms=timer(lambda:base(x))
    # Keep the same quantization for every backend and tile size.
    candidates={}
    for bk in GOOD_BK:
        module.set_backend("triton_tiled");module.tile_k=bk
        try:
            _=module(x)
            candidates[f"tiled_BK{bk}"]=timer(lambda:module(x))
        except Exception as exc:
            print("Tiled failure",bk,indim,outdim,repr(exc)[:200])
    module.set_backend("triton")
    try:
        candidates["legacy_triton"]=timer(lambda:module(x))
    except Exception as exc:
        print("Legacy packed kernel failure:",repr(exc)[:200])
    module.set_backend("cached_fp16");_=module(x)
    cachedms=timer(lambda:module(x))
    for backend,ms in candidates.items():
        micro.append(dict(in_features=indim,out_features=outdim,M=m,
                          backend=backend,ms=ms,original_fp16_ms=fp16ms,
                          original_over_candidate_speedup=fp16ms/ms))
    micro.append(dict(in_features=indim,out_features=outdim,M=m,
                      backend="cached_fp16",ms=cachedms,
                      original_fp16_ms=fp16ms,
                      original_over_candidate_speedup=fp16ms/cachedms))
    print(f"{indim}->{outdim} M={m}: FP16 {fp16ms:.3f} ms |",
          sorted(candidates.items(),key=lambda z:z[1])[:2],
          f"| cached {cachedms:.3f} ms")
    del base,module,x
    gc.collect();torch.cuda.empty_cache()
MICRO=pd.DataFrame(micro)
display(MICRO.sort_values(["in_features","out_features","ms"]))
MICRO.to_csv(OUT/"microbenchmark_v2.csv",index=False)

# Select fastest CORRECT packed candidate based solely on microbenchmark, never task test.
packed_only=MICRO[MICRO.backend!="cached_fp16"]
wins=packed_only.groupby("backend").apply(
    lambda grp:float(np.median(grp.original_over_candidate_speedup)),
    include_groups=False).sort_values(ascending=False)
CHOSEN_KERNEL=str(wins.index[0])
print("Best packed backend by median relative micro speed:",CHOSEN_KERNEL,
      "speedup",float(wins.iloc[0]))
# It might still be slower than the original FP16 reference!


In [ ]:
#@title 6. Real typed-decision calibration / development / audit data: case-disjoint
from datasets import load_dataset
def jsonish(value):
    if isinstance(value,str):
        try:return json.loads(value)
        except Exception:return value
    return value

def parse_items(rows,limit):
    groups=[]
    for row in rows:
        state=jsonish(row.get("state"))
        queries=jsonish(row.get("questions"))
        if not isinstance(queries,dict):continue
        this_case=[]
        for raw in queries.values():
            try:
                question=source.normalize_question(jsonish(raw))
                ids,marks=source.build_sequence(
                    tokenizer,state,question,max_len=MAX_LEN,head_max_len=192,
                    truncate_left=isinstance(state,list))
                if len(marks)<2:continue
                this_case.append(dict(ids=ids,markers=marks,qtype=source.QTYPES[question["t"]]))
            except (ValueError,TypeError,KeyError,IndexError):continue
        if this_case:groups.append(this_case)
    # Round-robin source cases; avoid first few cases dominating each partition.
    result=[]
    for depth in range(max(map(len,groups),default=0)):
        for group in groups:
            if depth<len(group):
                result.append(group[depth])
                if len(result)>=limit:return result
    return result

rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split=f"train[:{TRAIN_CASES}]"))
assert len(rows)>=300, "Insufficient training cases for disjoint partitions"
random.Random(SEED).shuffle(rows)
cal_rows=rows[:len(rows)//4]
dev_rows=rows[len(rows)//4:3*len(rows)//4]
audit_rows=rows[3*len(rows)//4:]
cal_items=parse_items(cal_rows,CAL_DECISIONS)
dev_items=parse_items(dev_rows,DEV_DECISIONS)
audit_items=parse_items(audit_rows,AUDIT_DECISIONS)
assert len(cal_items)==CAL_DECISIONS and len(dev_items)==DEV_DECISIONS
assert len(audit_items)==AUDIT_DECISIONS

def batchify(items,size=4,pad_len=None):
    output=[]
    for i in range(0,len(items),size):
        batch=source.collate_items(items[i:i+size],int(tokenizer.pad_token_id))
        if pad_len is not None:
            length=batch["input_ids"].shape[1]
            if length>pad_len:raise ValueError(f"Input requires {length} > requested {pad_len}")
            missing=pad_len-length
            if missing:
                batch["input_ids"]=F.pad(batch["input_ids"],(0,missing),
                                          value=int(tokenizer.pad_token_id))
                batch["attention_mask"]=F.pad(batch["attention_mask"],(0,missing),value=0)
        output.append({name:val.cuda() for name,val in batch.items()})
    return output

CAL=batchify(cal_items)
DEV=batchify(dev_items)
AUDIT=batchify(audit_items)
print("Disjoint train-split decision counts:",len(cal_items),len(dev_items),len(audit_items))


In [ ]:
#@title 7. Collect original teacher predictions, define cumulative fidelity gates
@torch.inference_mode()
def infer(model,batch):
    with torch.autocast("cuda",dtype=torch.float16):
        result=model(**batch)
    return result[0] if isinstance(result,(tuple,list)) else result

@torch.inference_mode()
def outputs(model,batches):
    return [infer(model,b).detach().float().cpu() for b in batches]

DEV_BASE=outputs(reference,DEV)
AUDIT_BASE=outputs(reference,AUDIT)

def compare(model,batches,teacher):
    n=matches=0;js_sum=0.0;max_logit=0.0
    with torch.inference_mode():
        for b,t in zip(batches,teacher):
            candidate=infer(model,b).float().cpu()
            mask=b["marker_mask"].cpu()
            candidate=candidate.masked_fill(~mask,-1e4)
            t=t.masked_fill(~mask,-1e4)
            p=candidate.softmax(-1).clamp_min(1e-9)
            q=t.softmax(-1).clamp_min(1e-9)
            avg=0.5*(p+q)
            js=0.5*((p*(p/avg).log()).sum(-1)+(q*(q/avg).log()).sum(-1))
            n+=len(t);matches+=int((p.argmax(-1)==q.argmax(-1)).sum())
            js_sum+=float(js.sum())
            max_logit=max(max_logit,float((candidate-t).abs().max()))
    return dict(n=n,agreement=matches/n,mean_js=js_sum/n,max_logit_abs=max_logit)

print("FP16 self-consistency DEV",compare(reference,DEV,DEV_BASE))
print("FP16 self-consistency AUDIT",compare(reference,AUDIT,AUDIT_BASE))


In [ ]:
#@title 8. One-layer-at-a-time: activation-aware scale refitting, then quality-gated conversion
import copy
candidate=copy.deepcopy(reference).eval().requires_grad_(False)
original_state_bytes=fast.state_bytes(reference)
names_accepted=[];rows=[];calibration_rows=[]
range_end=min(LAYER_COUNT,FIRST_LAYER+MAX_LAYERS)
for layer_no in range(FIRST_LAYER,range_end):
    prefix=f"encoder.layers.{layer_no}."
    names=[name for name,module in candidate.named_modules()
           if name.startswith(prefix) and isinstance(module,nn.Linear)
           and module.weight.numel()>=MIN_LINEAR_ELEMENTS]
    names.sort(key=lambda name:-candidate.get_submodule(name).weight.numel())
    if not names:continue
    print(f"\nLayer {layer_no:02d}: {len(names)} FP16 modules, calibration in present accumulated model")
    activations=fast.collect_calibration_inputs(
        candidate,names,CAL[:8],max_tokens=CALIBRATION_TOKENS_PER_PROJECTION)
    for name in names:
        old_linear=candidate.get_submodule(name)
        mod=fast.TiledTernaryLinear.from_linear(
            old_linear,backend="triton_tiled",tile_k=128)
        if ENABLE_SCALE_REFINEMENT:
            diag=fast.refine_scales_from_activations(
                mod,old_linear,activations[name],relative_ridge=RIDGE_RELATIVE)
        else:
            diag=dict(baseline_nmse=None,candidate_nmse=None,accepted=False)
        calibration_rows.append(dict(layer=layer_no,module=name,**diag))
        if CHOSEN_KERNEL.startswith("tiled_BK"):
            mod.set_backend("triton_tiled")
            mod.tile_k=int(CHOSEN_KERNEL.replace("tiled_BK",""))
        else:
            mod.set_backend("triton")
        fast.replace(candidate,name,mod)
        try:
            quality=compare(candidate,DEV,DEV_BASE)
            saved=fast.state_bytes(old_linear)-fast.state_bytes(mod)
            keep=(quality["agreement"]>=MIN_DEV_AGREEMENT
                  and quality["mean_js"]<=MAX_MEAN_JS and saved>0)
            if keep:
                names_accepted.append(name)
            else:
                fast.replace(candidate,name,old_linear)
            entry=dict(layer=layer_no,module=name,accepted=bool(keep),
                       saved_mib=saved/2**20,**quality)
            rows.append(entry)
            print(f'  {name.rsplit(".",2)[-2:]}: '
                  f'agree={quality["agreement"]:.2%} '
                  f'JS={quality["mean_js"]:.5f} '
                  f'cal-refit={diag["accepted"]} '
                  f'save={saved/2**20:.2f}MiB '
                  f'{"KEEP" if keep else "REVERT"}')
        except Exception as exc:
            fast.replace(candidate,name,old_linear)
            rows.append(dict(layer=layer_no,module=name,accepted=False,error=repr(exc)))
            print("  ERROR and rollback",name,repr(exc)[:300])
        pd.DataFrame(rows).to_csv(OUT/"layerwise_trials_v2.csv",index=False)
        pd.DataFrame(calibration_rows).to_csv(OUT/"activation_scale_refit.csv",index=False)
        (OUT/"accepted_modules_v2.json").write_text(json.dumps(names_accepted,indent=2))
    del activations
    gc.collect();torch.cuda.empty_cache()
    print("Cumulative converted modules:",len(names_accepted),"| state MiB:",
          round(fast.state_bytes(candidate)/2**20,2))
trials=pd.DataFrame(rows)
display(trials)


In [ ]:
#@title 9. Independent audit and one-time official test evaluation (NOT used for selection)
final_dev=compare(candidate,DEV,DEV_BASE)
final_audit=compare(candidate,AUDIT,AUDIT_BASE)
print("Selection DEV:",final_dev)
print("Independent train-case AUDIT:",final_audit)
AUDIT_GATE=(final_audit["agreement"]>=MIN_AUDIT_AGREEMENT
            and final_audit["mean_js"]<=MAX_MEAN_JS)
print("Independent quality gate", "PASS" if AUDIT_GATE else "FAIL")
official={}
if OFFICIAL_TEST_DECISIONS:
    test_rows=load_dataset("LocalLLaMA/typed-decisions","all",split="test")
    test_items=parse_items(test_rows,OFFICIAL_TEST_DECISIONS)
    if not test_items:raise RuntimeError("Unable to parse official test decisions")
    T_BATCH=batchify(test_items)
    test_teacher=outputs(reference,T_BATCH)
    official=compare(candidate,T_BATCH,test_teacher)
    print("Official TEST (not used for selection):",official)


In [ ]:
#@title 10. Real end-to-end GPU speed on 128, 256 and 512 tokens
END_TO_END=[]
for batch_n in (1,4):
    selected=dev_items[:batch_n]
    shortest=max(len(item["ids"]) for item in selected)
    for length in (128,256,512):
        if length<shortest or length>MAX_LEN:continue
        b=batchify(selected,size=batch_n,pad_len=length)[0]
        baseline=timer(lambda:infer(reference,b))
        for backend in (CHOSEN_KERNEL,"cached_fp16"):
            for m in candidate.modules():
                if isinstance(m,fast.TiledTernaryLinear):
                    if backend.startswith("tiled_BK"):
                        m.set_backend("triton_tiled")
                        m.tile_k=int(backend.removeprefix("tiled_BK"))
                    else:
                        m.set_backend('triton' if backend=='legacy_triton' else backend)
            _=infer(candidate,b)
            measured=timer(lambda:infer(candidate,b))
            END_TO_END.append(dict(batch=batch_n,tokens=length,backend=backend,
                fp16_ms=baseline,experimental_ms=measured,
                speedup=baseline/measured,
                packed_state_mib=fast.state_bytes(candidate)/2**20,
                actual_resident_weight_mib=fast.runtime_bytes(candidate)/2**20))
        print(f"B{batch_n} T{length}: FP16 {baseline:.2f} ms")
bench=pd.DataFrame(END_TO_END)
display(bench)
bench.to_csv(OUT/"end_to_end_v2.csv",index=False)
# Restore compressed backend before saving and judging.
for m in candidate.modules():
    if isinstance(m,fast.TiledTernaryLinear):
        if CHOSEN_KERNEL.startswith("tiled_BK"):
            m.set_backend("triton_tiled")
            m.tile_k=int(CHOSEN_KERNEL.removeprefix("tiled_BK"))
        else:m.set_backend("triton")
torch.cuda.empty_cache()


In [ ]:
#@title 11. Verdict: quality + packed speed, not just 2-bit size
packed_bench=bench[bench.backend==CHOSEN_KERNEL]
all_speedups=packed_bench.speedup.tolist()
best_speedup=max(all_speedups) if all_speedups else None
all_faster=bool(all_speedups) and min(all_speedups)>1
some_faster=bool(all_speedups) and max(all_speedups)>1
result={
    "checkpoint":SOURCE_REPO,
    "gpu":torch.cuda.get_device_name(0),
    "precision":"ternary packed 2-bit codes G128 FP16 scales, FP16 tensor-core, FP32 acc",
    "physical_bits_per_converted_weight":2.125,
    "first_layer":FIRST_LAYER,"max_layers":MAX_LAYERS,
    "converted_modules":names_accepted,
    "original_state_mib":original_state_bytes/2**20,
    "compressed_state_mib":fast.state_bytes(candidate)/2**20,
    "compressed_reduction_percent":100*(1-fast.state_bytes(candidate)/original_state_bytes),
    "best_packed_backend_by_microbenchmark":CHOSEN_KERNEL,
    "packed_speedup_best":best_speedup,
    "packed_speedup_all_benchmark_shapes":all_speedups,
    "faster_in_any_shape":some_faster,
    "faster_in_every_shape":all_faster,
    "selection":final_dev,"audit":final_audit,"official_test":official,
    "independent_quality_pass":bool(AUDIT_GATE),
    "ready_to_replace_original":bool(AUDIT_GATE and all_faster and bool(names_accepted)),
    "notes":[
      "No training on official test examples",
      "Cached FP16 loses resident weight-compression benefit",
      "Model selection uses train-split development cases",
      "This is PTQ ternary + activation-aware scale fitting, not native trained BitNet",
      "No source checkpoint was modified or published"]
}
(OUT/"summary_v2.json").write_text(json.dumps(result,indent=2))
print(json.dumps({k:result[k] for k in (
    "compressed_reduction_percent","best_packed_backend_by_microbenchmark",
    "packed_speedup_best","selection","audit","official_test",
    "ready_to_replace_original")},indent=2))


In [ ]:
#@title 12. Write portable *delta-only* compressed checkpoint (source remains unchanged)
from safetensors.torch import save_file
if names_accepted:
    delta={}
    for name in names_accepted:
        mod=candidate.get_submodule(name)
        for suffix,value in mod.state_dict().items():
            delta[f"{name}.{suffix}"]=value.detach().contiguous().cpu()
    save_file(delta,str(OUT/"laya_v2_ternary_delta.safetensors"))
    (OUT/"delta_manifest.json").write_text(json.dumps({
        "source_repo":SOURCE_REPO,
        "required_base":"exact FP16 original model, same revision",
        "modules":names_accepted,
        "g128_scale_fit":ENABLE_SCALE_REFINEMENT,
        "triton_backend":CHOSEN_KERNEL,
        "format":"2-bit ternary codes plus G128 FP16 row-scales",
        "warning":"Experimental weight delta, not a self-contained HF model"},indent=2))
    print("Saved converted-weight delta bytes:",
          (OUT/"laya_v2_ternary_delta.safetensors").stat().st_size)
else:
    print("No projections met development gate. Saving diagnostics only.")


In [ ]:
#@title 13. Export results ZIP for next iteration
outzip=OUT/"Laya_Tiled_Ternary_V2_Results.zip"
with zipfile.ZipFile(outzip,"w",zipfile.ZIP_DEFLATED) as z:
    for item in OUT.iterdir():
        if item.is_file() and item!=outzip:
            z.write(item,item.name)
print("Output:",outzip)
try:
    from google.colab import files
    files.download(str(outzip))
except ImportError:
    pass


## How to read this experiment

- **Tiled BK vs legacy:** this isolates the effect of scaling each decoded weight *before* FP16 tensor-core multiplication and using larger K tiles.
- **Tiled vs cached FP16:** the speed gap quantifies the unpacking/dequant overhead. The cached baseline has FP16 reconstructed weights resident and is **not compressed in active GPU memory**.
- **Scales:** G128 weights remain strict {-1,0,+1}, and the activation-aware refit changes only the group scales. It is much cheaper than full STE/QAT but is not guaranteed to pass fidelity.
- **Selection vs audit:** converted projections are selected only on one train-case subset; audit and official test are separate. If audit fails, do not promote the checkpoint.
- **Hardware:** T4 and L4 have different kernel sweet spots. A speedup must be established for the target GPU and actual batch/sequence shapes.

The v2 notebook does not train the whole model, publish to Hugging Face, or overwrite the source model. It saves a compact delta plus CSV/JSON evidence for further review.
